In [60]:
!pip install plotnine litellm datasets


In [61]:
import logging
import re
import pandas as pd
from plotnine import ggplot, aes, geom_line, labs, theme_minimal
from typing import Any, Dict
from langchain_community.chat_models import ChatLiteLLM
from langchain_core.messages import HumanMessage

In [62]:
# Minimal logger
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [63]:
# ─── Cell 2: Credentials & LLM Initialization ─────────────────────────────────
API_KEY  = "sk-i1pU6I-3YVuwt20ZV2wJHg"
API_BASE = "https://llm-api.cyverse.ai/v1"
MODEL_ID = "litellm_proxy/Meta-Llama-3.1-70B-Instruct-quantized"

logger.info("Loading LLM %s …", MODEL_ID)
llm = ChatLiteLLM(
    model=MODEL_ID,
    api_key=API_KEY,
    api_base=API_BASE,
    temperature=0
)
logger.info("LLM ready")

INFO:__main__:Loading LLM litellm_proxy/Meta-Llama-3.1-70B-Instruct-quantized …
INFO:__main__:LLM ready


In [64]:
def analyze_df(df: pd.DataFrame, question: str, sample_rows: int = 200) -> str:
    if df.empty:
        return "❌ DataFrame is empty."
    snippet = df.head(sample_rows).to_csv(index=False)
    correction_clause = "Also, validate this against the full dataset and correct any mistaken assumptions."
    prompt = f"""
You are a highly-skilled data storyteller and senior analyst. Here's a preview of a dataset:

{snippet}

QUESTION: {question}

Please:
- Answer ONLY the specific question
- Point out unique/non-obvious insights
- Avoid repeating basic column summaries
- Use concrete stats or comparisons when possible
- Bullet your answer for clarity and impact
- {correction_clause}

Respond with only valid, evidence-based bullet points.
You must only report facts verifiable from the full dataset. Never guess or inflate stats. If something can't be derived, say so explicitly.


"""
    logger.info("Asking refined analysis question…")
    resp = llm.invoke([HumanMessage(content=prompt)])
    return clean_markdown(resp.content)

In [65]:
def describe_plot(df: pd.DataFrame, code: str, sample_rows: int = 100) -> str:
    snippet = df.head(sample_rows).to_csv(index=False)
    prompt = f"""
You are an expert data visual analyst. A user has generated the following plot using Python and plotnine:

{code}

Here are the first {sample_rows} rows of the data:
{snippet}

Write a human-style narrative caption that explains:
- What the plot shows
- What unique insights it reveals
- What patterns or anomalies are evident
- Why it's important

Be concise and smart. Use no fluff. Return as bullet points.
"""
    logger.info("Requesting narrative description for plot…")
    resp = llm.invoke([HumanMessage(content=prompt)])
    return clean_markdown(resp.content)

In [66]:
def visualize_df(df: pd.DataFrame, goal: str, sample_rows: int = 500) -> str:
    if df.empty:
        return ""
    snippet = df.head(sample_rows).to_csv(index=False)
    prompt = f"""
You have a pandas DataFrame named `df`. Here's a preview of the first {sample_rows} rows:
{snippet}

GOAL: {goal}

Write working Python code using plotnine that:
- Uses only the required imports (ggplot, aes, geom_*, labs, theme_minimal)
- Creates an insightful plot from `df`
- Assigns to `plot` and ends with print(plot)
- Avoids markdown or explanations

Your code must be VALID and runnable.
"""
    logger.info("Requesting code for data visualization…")
    resp = llm.invoke([HumanMessage(content=prompt)])
    return clean_markdown(resp.content)


In [67]:
def run_code_snippet(code: str, extra_globals: Dict[str, Any] = None) -> None:
    base = {
        "pd": pd,
        "df": df,
        "ggplot": ggplot,
        "aes": aes,
        "geom_line": geom_line,
        "geom_point": geom_point,
        "geom_bar": geom_bar,
        "labs": labs,
        "theme_minimal": theme_minimal,
    }
    if extra_globals:
        base.update(extra_globals)
    exec(code, base)


In [68]:
from datasets import load_dataset
import pandas as pd

# Load movie dataset
movies = load_dataset("Pablinho/movies-dataset", split="train")
df = pd.DataFrame(movies)

In [69]:
df.head()

,Release_Date,Title,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Poster_Url
0,2021-12-15,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,5083.954,8940,8.3,en,"Action, Adventure, Science Fiction",https://image.tmdb.org/t/p/original/1g0dhYtq4i...
1,2022-03-01,The Batman,"In his second year of fighting crime, Batman u...",3827.658,1151,8.1,en,"Crime, Mystery, Thriller",https://image.tmdb.org/t/p/original/74xTEgt7R3...
2,2022-02-25,No Exit,Stranded at a rest stop in the mountains durin...,2618.087,122,6.3,en,Thriller,https://image.tmdb.org/t/p/original/vDHsLnOWKl...
3,2021-11-24,Encanto,"The tale of an extraordinary family, the Madri...",2402.201,5076,7.7,en,"Animation, Comedy, Family, Fantasy",https://image.tmdb.org/t/p/original/4j0PNHkMr5...
4,2021-12-22,The King's Man,As a collection of history's worst tyrants and...,1895.511,1793,7.0,en,"Action, Adventure, Thriller, War",https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...


In [70]:
df.to_csv('data.csv')

In [73]:
question = "Which movie has more popularity?"
print("=== Question ===")
print(question)
print("=== Analysis ===")
print(analyze_df(df, question))


INFO:__main__:Asking refined analysis question…
20:30:53 - LiteLLM:INFO: utils.py:3119 - 
LiteLLM completion() model= Meta-Llama-3.1-70B-Instruct-quantized; provider = litellm_proxy
INFO:LiteLLM:
LiteLLM completion() model= Meta-Llama-3.1-70B-Instruct-quantized; provider = litellm_proxy


=== Question ===
Which movie has more popularity?
=== Analysis ===


INFO:httpx:HTTP Request: POST https://llm-api.cyverse.ai/v1/chat/completions "HTTP/1.1 200 OK"
20:31:01 - LiteLLM:INFO: utils.py:1215 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler


The movie "Spider-Man: No Way Home" has the highest popularity with a value of 5083.954.
In comparison, the second most popular movie "The Batman" has a popularity of 3827.658, which is approximately 24.6% lower than "Spider-Man: No Way Home".
The top 5 most popular movies are all English-language films, with "Spider-Man: No Way Home", "The Batman", "No Exit", "Encanto", and "The King's Man" having popularity values of 5083.954, 3827.658, 2618.087, 2402.201, and 1895.511 respectively.
The average popularity of the top 10 most popular movies is approximately 2831.119, which is significantly higher than the overall average popularity of the dataset.
The dataset suggests that English-language Action, Adventure, and Science Fiction movies tend to have higher popularity values compared to other genres and languages.


In [74]:
import pandas as pd

# Ensure 'Popularity' column is numeric
df["Popularity"] = pd.to_numeric(df["Popularity"], errors="coerce")

# 1. Top movie by popularity
top_movie = df.loc[df["Popularity"].idxmax()]
print("🎬 Most Popular Movie:")
print(top_movie[["Title", "Popularity"]])

# 2. Top 5 most popular movies
top5 = df.sort_values("Popularity", ascending=False).head(5)
print("\n🔥 Top 5 Most Popular Movies:")
print(top5[["Title", "Popularity"]])

# 3. Top 10 average popularity
top10_avg = df.sort_values("Popularity", ascending=False).head(10)["Popularity"].mean()
print("\n📊 Average Popularity of Top 10 Movies:", round(top10_avg, 3))

# 4. Overall dataset popularity average
overall_avg = df["Popularity"].mean()
print("📉 Overall Average Popularity:", round(overall_avg, 3))

# 5. Language distribution in top 5
top5_langs = df.loc[top5.index, "Original_Language"].value_counts()
print("\n🌍 Language Distribution in Top 5 Movies:")
print(top5_langs)

# 6. Genre overview (if n


🎬 Most Popular Movie:
Title         Spider-Man: No Way Home
Popularity                   5083.954
Name: 0, dtype: object

🔥 Top 5 Most Popular Movies:
                     Title  Popularity
0  Spider-Man: No Way Home    5083.954
1               The Batman    3827.658
2                  No Exit    2618.087
3                  Encanto    2402.201
4           The King's Man    1895.511

📊 Average Popularity of Top 10 Movies: 2398.626
📉 Overall Average Popularity: 40.321

🌍 Language Distribution in Top 5 Movies:
Original_Language
en    5
Name: count, dtype: int64


In [78]:
question = "Which original languages people watch more movies?"
print("=== Question ===")
print(question)
print("=== Analysis ===")
print(analyze_df(df, question))

INFO:__main__:Asking refined analysis question…
20:42:42 - LiteLLM:INFO: utils.py:3119 - 
LiteLLM completion() model= Meta-Llama-3.1-70B-Instruct-quantized; provider = litellm_proxy
INFO:LiteLLM:
LiteLLM completion() model= Meta-Llama-3.1-70B-Instruct-quantized; provider = litellm_proxy


=== Question ===
Which original languages people watch more movies?
=== Analysis ===


INFO:httpx:HTTP Request: POST https://llm-api.cyverse.ai/v1/chat/completions "HTTP/1.1 200 OK"
20:42:48 - LiteLLM:INFO: utils.py:1215 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler


The most popular original language for movies is English, with 148 movies, accounting for approximately 74% of the total dataset.
The second most popular original language is Japanese, with 6 movies, accounting for around 3% of the total dataset.
Other languages, such as French, Spanish, German, Italian, and others, have fewer than 5 movies each, indicating a significant dominance of English-language content in the dataset.
The top 3 languages (English, Japanese, and French) account for around 80% of the total movies, while the remaining 20% is spread across 10 other languages.
It's worth noting that the dataset may be biased towards English-language movies, and the results may not be representative of global movie-watching habits.
